# Calendar Dimension Table

Builds the standard calendar dimension `adb_caresyncs.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31, including year, quarter, month, week, day and flag attributes.

In [0]:
%sql
-- Standard calendar dimension: one row per day, 1940-01-01 to 2030-12-31
CREATE OR REPLACE TABLE adb_caresyncs.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS calendar_date
)
SELECT
  CAST(date_format(calendar_date, 'yyyyMMdd') AS INT)        AS date_key,
  calendar_date,
  -- Year
  year(calendar_date)                                         AS year,
  CAST(date_trunc('YEAR', calendar_date) AS DATE)             AS year_start_date,
  CAST(make_date(year(calendar_date), 12, 31) AS DATE)        AS year_end_date,
  CASE WHEN (year(calendar_date) % 4 = 0 AND year(calendar_date) % 100 <> 0)
         OR year(calendar_date) % 400 = 0 THEN true ELSE false END AS is_leap_year,
  -- Half year
  CASE WHEN month(calendar_date) <= 6 THEN 1 ELSE 2 END       AS half_year,
  concat('H', CASE WHEN month(calendar_date) <= 6 THEN 1 ELSE 2 END) AS half_year_name,
  -- Quarter
  quarter(calendar_date)                                      AS quarter,
  concat('Q', quarter(calendar_date))                         AS quarter_name,
  concat(year(calendar_date), '-Q', quarter(calendar_date))   AS year_quarter,
  CAST(date_trunc('QUARTER', calendar_date) AS DATE)          AS quarter_start_date,
  date_sub(add_months(CAST(date_trunc('QUARTER', calendar_date) AS DATE), 3), 1) AS quarter_end_date,
  datediff(calendar_date, CAST(date_trunc('QUARTER', calendar_date) AS DATE)) + 1 AS day_of_quarter,
  -- Month
  month(calendar_date)                                        AS month,
  date_format(calendar_date, 'MMMM')                          AS month_name,
  date_format(calendar_date, 'MMM')                           AS month_short_name,
  date_format(calendar_date, 'yyyy-MM')                       AS year_month,
  CAST(date_format(calendar_date, 'yyyyMM') AS INT)           AS year_month_key,
  CAST(date_trunc('MONTH', calendar_date) AS DATE)            AS month_start_date,
  last_day(calendar_date)                                     AS month_end_date,
  day(last_day(calendar_date))                                AS days_in_month,
  -- Week (ISO 8601: weeks start Monday, week 1 contains first Thursday)
  weekofyear(calendar_date)                                   AS week_of_year,
  extract(YEAROFWEEK FROM calendar_date)                      AS iso_year,
  concat(extract(YEAROFWEEK FROM calendar_date), '-W', lpad(weekofyear(calendar_date), 2, '0')) AS iso_year_week,
  CAST(ceil(day(calendar_date) / 7.0) AS INT)                 AS week_of_month,
  date_sub(calendar_date, weekday(calendar_date))             AS week_start_date,
  date_add(calendar_date, 6 - weekday(calendar_date))         AS week_end_date,
  -- Day
  day(calendar_date)                                          AS day_of_month,
  dayofyear(calendar_date)                                    AS day_of_year,
  weekday(calendar_date) + 1                                  AS day_of_week,       -- 1 = Monday ... 7 = Sunday (ISO)
  dayofweek(calendar_date)                                    AS day_of_week_sun_start, -- 1 = Sunday ... 7 = Saturday
  date_format(calendar_date, 'EEEE')                          AS day_name,
  date_format(calendar_date, 'EEE')                           AS day_short_name,
  -- Flags
  weekday(calendar_date) >= 5                                 AS is_weekend,
  weekday(calendar_date) < 5                                  AS is_weekday,
  calendar_date = CAST(date_trunc('MONTH', calendar_date) AS DATE) AS is_month_start,
  calendar_date = last_day(calendar_date)                     AS is_month_end,
  calendar_date = CAST(date_trunc('QUARTER', calendar_date) AS DATE) AS is_quarter_start,
  calendar_date = date_sub(add_months(CAST(date_trunc('QUARTER', calendar_date) AS DATE), 3), 1) AS is_quarter_end,
  (month(calendar_date) = 1 AND day(calendar_date) = 1)       AS is_year_start,
  (month(calendar_date) = 12 AND day(calendar_date) = 31)     AS is_year_end
FROM dates
ORDER BY calendar_date;

In [0]:
%sql
-- Validation: row count, date range, uniqueness, leap days
SELECT
  count(*)                                   AS total_rows,
  count(DISTINCT calendar_date)              AS distinct_dates,
  min(calendar_date)                         AS min_date,
  max(calendar_date)                         AS max_date,
  datediff(DATE'2030-12-31', DATE'1940-01-01') + 1 AS expected_rows,
  sum(CASE WHEN month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS leap_days
FROM adb_caresyncs.silver.calendar;

In [0]:
%sql
SELECT * FROM adb_caresyncs.silver.calendar
WHERE calendar_date BETWEEN DATE'2024-12-28' AND DATE'2025-01-06'
ORDER BY calendar_date;